In [2]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

df = pd.read_csv('./data/application_train.csv')
print(df.shape)

(307511, 122)


In [3]:
# Corrections
df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)
df = df[df['CODE_GENDER'] != 'XNA']

# Features dérivées
df['CREDIT_INCOME_RATIO'] = df['AMT_CREDIT'] / df['AMT_INCOME_TOTAL']
df['ANNUITY_INCOME_RATIO'] = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']
df['EMPLOYED_TO_AGE_RATIO'] = df['DAYS_EMPLOYED'] / df['DAYS_BIRTH']
df['CREDIT_TERM'] = df['AMT_ANNUITY'] / df['AMT_CREDIT']
df['INCOME_PER_PERSON'] = df['AMT_INCOME_TOTAL'] / df['CNT_FAM_MEMBERS']

# Drop colonnes +40% missing sauf EXT_SOURCE_1
missing_pct = df.isnull().mean() * 100
cols_to_drop = missing_pct[missing_pct > 40].index.tolist()
cols_to_drop = [c for c in cols_to_drop if c != 'EXT_SOURCE_1']
df = df.drop(columns=cols_to_drop)

# Drop colonnes non pertinentes
df = df.drop(columns=['SK_ID_CURR', 'WEEKDAY_APPR_PROCESS_START'])

X = df.drop(columns=['TARGET'])
y = df['TARGET']

print(X.shape)

(307507, 76)


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import FunctionTransformer
import xgboost as xgb

# Identifier les colonnes
num_cols = X.select_dtypes(include=['float64', 'int64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

# Pipeline numérique — imputation médiane + winsorization
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
])

# Pipeline catégoriel — imputation + one-hot encoding
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combiner
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

print("Preprocessor défini ✓")

Preprocessor défini ✓


In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', xgb.XGBClassifier(
        n_estimators=494,
        max_depth=4,
        learning_rate=0.086,
        subsample=0.786,
        colsample_bytree=0.932,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        eval_metric='auc'
    ))
])

full_pipeline.fit(X_train, y_train)
print("Pipeline entraîné ✓")

Pipeline entraîné ✓


In [6]:
from sklearn.metrics import roc_auc_score
from scipy.stats import ks_2samp

y_pred = full_pipeline.predict_proba(X_test)[:, 1]

auc = roc_auc_score(y_test, y_pred)
gini = 2 * auc - 1
ks, _ = ks_2samp(y_pred[y_test == 1], y_pred[y_test == 0])

print(f"AUC-ROC : {auc:.4f}")
print(f"Gini    : {gini:.4f}")
print(f"KS      : {ks:.4f}")

AUC-ROC : 0.7703
Gini    : 0.5405
KS      : 0.4024


In [7]:
import pickle

with open('pipeline.pkl', 'wb') as f:
    pickle.dump(full_pipeline, f)

print("Pipeline sauvegardé ✓")

Pipeline sauvegardé ✓


In [9]:
test = pd.read_csv('./data/application_test.csv')
test_ids = test['SK_ID_CURR']

# Mêmes corrections
test['DAYS_EMPLOYED'] = test['DAYS_EMPLOYED'].replace(365243, np.nan)
test = test.drop(columns=cols_to_drop, errors='ignore')
test = test.drop(columns=['SK_ID_CURR', 'WEEKDAY_APPR_PROCESS_START'], errors='ignore')

# Features dérivées
test['CREDIT_INCOME_RATIO'] = test['AMT_CREDIT'] / test['AMT_INCOME_TOTAL']
test['ANNUITY_INCOME_RATIO'] = test['AMT_ANNUITY'] / test['AMT_INCOME_TOTAL']
test['EMPLOYED_TO_AGE_RATIO'] = test['DAYS_EMPLOYED'] / test['DAYS_BIRTH']
test['CREDIT_TERM'] = test['AMT_ANNUITY'] / test['AMT_CREDIT']
test['INCOME_PER_PERSON'] = test['AMT_INCOME_TOTAL'] / test['CNT_FAM_MEMBERS']

# Le Pipeline gère le reste automatiquement
y_pred_test = full_pipeline.predict_proba(test)[:, 1]

submission = pd.DataFrame({
    'SK_ID_CURR': test_ids,
    'TARGET': y_pred_test
})

submission.to_csv('data/clean/submission_pipeline.csv', index=False)
print(submission.head())

   SK_ID_CURR    TARGET
0      100001  0.186937
1      100005  0.600514
2      100013  0.075707
3      100028  0.312109
4      100038  0.622293
